# 1) Validation Experiment Overview

This notebook implements the validation pipeline for the proposed model selection framework developed in the bachelor thesis.

The validation is performed on:

- **HTRU2** – binary classification
- **Diabetic Retinopathy Debrecen** – binary classification
- **Page Blocks** – multi-class classification
- **Glass Identification** – multi-class classification

The notebook performs the following steps:

1. dataset loading and inspection  
2. dataset characterization for framework application  
3. standard, defensible preprocessing  
4. hyperparameter optimization on the original validation datasets  
5. model training and evaluation using stratified 5-fold cross-validation  
6. comparison of observed best-performing models with the framework recommendations  

The following algorithms are evaluated:

- Decision Tree (CART)
- Random Forest
- Extra Trees
- Gradient Boosting
- XGBoost
- LightGBM
- CatBoost

The validation uses the same general methodological principles as the main thesis experiments:

- stratified 5-fold cross-validation
- binary main metric: F1-score
- multi-class main metric: Macro-F1
- additional metrics:
  - Precision / Recall
  - Macro-Precision / Macro-Recall
  - Balanced Accuracy
  - Training Time
  - Model Size

Unlike the main experiments, this validation notebook does **not** include:

- dataset size manipulation
- class imbalance manipulation
- Friedman test
- Nemenyi post-hoc analysis

This is because the purpose of the validation is to test whether the framework’s recommendations remain reasonable on unseen original datasets, rather than to construct new experimental conditions.

# 2) Project Structure and Execution

This notebook runs in Google Colab or a local Python environment.

When executed in Google Colab, the GitHub repository is cloned so that the notebook can access:

- validation datasets
- saved hyperparameter configuration files
- result output folders

The repository contains:

- `datasets/original_datasets`  
  Original datasets used in experiments and validation.

- `config`  
  Saved hyperparameter configuration files.

- `results`  
  Validation results exported by this notebook.

In [1]:

# Repository Setup

import os

REPO_NAME = "Tree-algorithms-dataset-characteristics"
REPO_URL = "https://github.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics.git"

if "COLAB_GPU" in os.environ and not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

if "COLAB_GPU" in os.environ and os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

# Paths used in the notebook

BASE_PATH = os.getcwd()

ORIGINAL_DATASETS_PATH = os.path.join(
    BASE_PATH, "datasets", "original_datasets"
)

CONFIG_PATH = os.path.join(
    BASE_PATH, "config"
)

RESULTS_PATH = os.path.join(
    BASE_PATH, "results"
)

os.makedirs(CONFIG_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("Repository ready.")
print("BASE_PATH:", BASE_PATH)
print("ORIGINAL_DATASETS_PATH:", ORIGINAL_DATASETS_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("RESULTS_PATH:", RESULTS_PATH)

/content/Tree-algorithms-dataset-characteristics
Repository ready.
BASE_PATH: /content/Tree-algorithms-dataset-characteristics
ORIGINAL_DATASETS_PATH: /content/Tree-algorithms-dataset-characteristics/datasets/original_datasets
CONFIG_PATH: /content/Tree-algorithms-dataset-characteristics/config
RESULTS_PATH: /content/Tree-algorithms-dataset-characteristics/results


# 3) Environment Setup and Library Versions

This section installs the required libraries and imports all packages used in the validation pipeline.

The libraries support:

- dataset loading
- preprocessing
- machine learning model training
- hyperparameter optimization
- evaluation and result aggregation

For reproducibility, the versions of the main libraries are also printed.

In [2]:
# Install required libraries (for Colab environment)
!pip install -q catboost lightgbm xgboost requests


# Imports
import json
import random
import warnings
import pickle
import requests

import pandas as pd
import numpy as np

from IPython.display import display

from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")


# Random Seed Configuration
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Random seed set to:", SEED)


# Global CV
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# Model Size Utility
def get_model_size_kb(model):
    return len(pickle.dumps(model)) / 1024

# Library versions
import sklearn
import scipy
import matplotlib

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)
print("catboost:", cb.__version__)
print("scipy:", scipy.__version__)
print("matplotlib:", matplotlib.__version__)

Random seed set to: 42
numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
xgboost: 3.2.0
lightgbm: 4.6.0
catboost: 1.2.10
scipy: 1.16.3
matplotlib: 3.10.0


# 4) Dataset Loading and Inspection

This section loads the original datasets used for framework validation and performs a basic inspection of their structure.

The datasets are stored in the repository directory:

`datasets/original_datasets`

Four validation datasets are used:

- **HTRU2** – a binary classification dataset containing pulsar candidate measurements.
- **Diabetic Retinopathy Debrecen** – a binary classification dataset with lesion-extraction features.
- **Page Blocks** – a multi-class classification dataset containing block-level page layout features.
- **Glass Identification** – a multi-class classification dataset with glass composition features.

After loading the datasets, a brief inspection is performed to verify:

- dataset dimensions
- feature data types
- missing values
- class distributions

Standard defensible cleaning is applied where needed:

- `?` values are converted to missing values
- all columns are converted to numeric where appropriate
- incomplete rows are removed
- identifier columns are removed when they are not predictors

In [3]:

# Dataset Loading (VALIDATION DATASETS)

# File paths
htru2_path = os.path.join(ORIGINAL_DATASETS_PATH, "HTRU_2.csv")
page_blocks_path = os.path.join(ORIGINAL_DATASETS_PATH, "page_blocks.csv")
retinopathy_path = os.path.join(ORIGINAL_DATASETS_PATH, "diabetic_retinopathy_debrecen.csv")
glass_path = os.path.join(ORIGINAL_DATASETS_PATH, "glass_identification.csv")

# --------------------------------------------------
# HTRU2 column names
# --------------------------------------------------

htru2_columns = [
    "Profile_mean",
    "Profile_stdev",
    "Profile_skewness",
    "Profile_kurtosis",
    "DM_mean",
    "DM_stdev",
    "DM_skewness",
    "DM_kurtosis",
    "class"
]

# --------------------------------------------------
# Load raw datasets
# --------------------------------------------------

htru2 = pd.read_csv(
    htru2_path,
    header=None,
    names=htru2_columns
)

page_blocks = pd.read_csv(page_blocks_path)
diabetic_retinopathy = pd.read_csv(retinopathy_path)
glass = pd.read_csv(glass_path)

# --------------------------------------------------
# Target detection utilities
# --------------------------------------------------

def find_binary_target_column(df):
    preferred = [
        "class", "Class", "target", "Target", "Outcome", "outcome",
        "label", "Label"
    ]

    for col in preferred:
        if col in df.columns and df[col].nunique(dropna=True) == 2:
            return col

    last_col = df.columns[-1]
    if df[last_col].nunique(dropna=True) == 2:
        return last_col

    binary_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 2]
    if len(binary_cols) == 1:
        return binary_cols[0]

    return None


def find_multiclass_target_column(df):
    preferred = ["class", "Class", "Type", "type", "target", "Target", "label", "Label"]

    for col in preferred:
        if col in df.columns and df[col].nunique(dropna=True) > 2:
            return col

    last_col = df.columns[-1]
    if df[last_col].nunique(dropna=True) > 2:
        return last_col

    return None

# --------------------------------------------------
# Standard cleaning
# --------------------------------------------------

# HTRU2
htru2 = htru2.replace("?", np.nan)
htru2 = htru2.apply(pd.to_numeric, errors="coerce")
htru2 = htru2.dropna().reset_index(drop=True)

# Page Blocks
page_blocks = page_blocks.replace("?", np.nan)
page_blocks = page_blocks.apply(pd.to_numeric, errors="coerce")
page_blocks = page_blocks.dropna().reset_index(drop=True)

# Diabetic Retinopathy Debrecen
diabetic_retinopathy = diabetic_retinopathy.replace("?", np.nan)
diabetic_retinopathy = diabetic_retinopathy.apply(pd.to_numeric, errors="coerce")
diabetic_retinopathy = diabetic_retinopathy.dropna().reset_index(drop=True)

# Glass Identification
glass = glass.replace("?", np.nan)
glass = glass.apply(pd.to_numeric, errors="coerce")
glass = glass.dropna().reset_index(drop=True)

# Drop Glass ID column if present
for col in ["Id_number", "id_number", "Id", "ID"]:
    if col in glass.columns:
        glass = glass.drop(columns=[col])
        break

# --------------------------------------------------
# Detect target columns
# --------------------------------------------------

htru2_target_col = "class"
page_target_col = find_multiclass_target_column(page_blocks)
dr_target_col = find_binary_target_column(diabetic_retinopathy)
glass_target_col = find_multiclass_target_column(glass)

if page_target_col is None:
    raise ValueError("Could not detect the target column in page_blocks.csv.")

if dr_target_col is None:
    raise ValueError("Could not detect a binary target column in diabetic_retinopathy_debrecen.csv.")

if glass_target_col is None:
    raise ValueError("Could not detect the target column in glass_identification.csv.")

# --------------------------------------------------
# Preview
# --------------------------------------------------

print("HTRU2 dataset preview:")
display(htru2.head())

print("\nPage Blocks dataset preview:")
display(page_blocks.head())

print("\nDiabetic Retinopathy Debrecen dataset preview:")
display(diabetic_retinopathy.head())

print("\nGlass Identification dataset preview:")
display(glass.head())


# Dataset Inspection Utility

def inspect_dataset(df, target, name):
    print("\n====================================")
    print(f"{name.upper()} DATASET")
    print("====================================")

    print("\nShape:", df.shape)

    print("\nFeature types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print(f"\nTarget distribution ({target}):")
    print(df[target].value_counts())

    print("\nClass proportions:")
    print(df[target].value_counts(normalize=True))


# Inspect cleaned datasets

inspect_dataset(htru2, htru2_target_col, "HTRU2")
inspect_dataset(page_blocks, page_target_col, "Page Blocks")
inspect_dataset(diabetic_retinopathy, dr_target_col, "Diabetic Retinopathy Debrecen")
inspect_dataset(glass, glass_target_col, "Glass Identification")

HTRU2 dataset preview:


,Profile_mean,Profile_stdev,Profile_skewness,Profile_kurtosis,DM_mean,DM_stdev,DM_skewness,DM_kurtosis,class
0,140.562500,55.683782,-0.234571,-0.699648,3.199833,19.110426,7.975532,74.242225,0
1,102.507812,58.882430,0.465318,-0.515088,1.677258,14.860146,10.576487,127.393580,0
2,103.015625,39.341649,0.323328,1.051164,3.121237,21.744669,7.735822,63.171909,0
3,136.750000,57.178449,-0.068415,-0.636238,3.642977,20.959280,6.896499,53.593661,0
4,88.726562,40.672225,0.600866,1.123492,1.178930,11.468720,14.269573,252.567306,0



Page Blocks dataset preview:


,height,length,area,eccen,p_black,p_and,mean_tr,blackpix,blackand,wb_trans,class
0,5,7,35,1.400,0.400,0.657,2.33,14,23,6,1
1,6,7,42,1.167,0.429,0.881,3.60,18,37,5,1
2,6,18,108,3.000,0.287,0.741,4.43,31,80,7,1
3,5,7,35,1.400,0.371,0.743,4.33,13,26,3,1
4,6,3,18,0.500,0.500,0.944,2.25,9,17,4,1



Diabetic Retinopathy Debrecen dataset preview:


,quality,pre_screening,ma1,ma2,ma3,ma4,ma5,ma6,exudate1,exudate2,exudate3,exudate3.1,exudate5,exudate6,exudate7,exudate8,macula_opticdisc_distance,opticdisc_diameter,am_fm_classification,Class
0,1,1,22,22,22,19,18,14,49.895756,17.775994,5.270920,5.270920,0.018632,0.006864,0.003923,0.003923,0.486903,0.100025,1,0
1,1,1,24,24,22,18,16,13,57.709936,23.799994,3.325423,3.325423,0.003903,0.003903,0.003903,0.003903,0.520908,0.144414,0,0
2,1,1,62,60,59,54,47,33,55.831441,27.993933,12.687485,12.687485,1.393889,0.373252,0.041817,0.007744,0.530904,0.128548,0,1
3,1,1,55,53,53,50,43,31,40.467228,18.445954,9.118901,9.118901,0.840261,0.272434,0.007653,0.001531,0.483284,0.114790,0,0
4,1,1,44,44,44,41,39,27,18.026254,8.570709,0.410381,0.410381,0.000000,0.000000,0.000000,0.000000,0.475935,0.123572,0,1



Glass Identification dataset preview:


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type_of_glass
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1



HTRU2 DATASET

Shape: (17898, 9)

Feature types:
Profile_mean        float64
Profile_stdev       float64
Profile_skewness    float64
Profile_kurtosis    float64
DM_mean             float64
DM_stdev            float64
DM_skewness         float64
DM_kurtosis         float64
class                 int64
dtype: object

Missing values:
Profile_mean        0
Profile_stdev       0
Profile_skewness    0
Profile_kurtosis    0
DM_mean             0
DM_stdev            0
DM_skewness         0
DM_kurtosis         0
class               0
dtype: int64

Target distribution (class):
class
0    16259
1     1639
Name: count, dtype: int64

Class proportions:
class
0    0.908426
1    0.091574
Name: proportion, dtype: float64

PAGE BLOCKS DATASET

Shape: (5473, 11)

Feature types:
height        int64
length        int64
area          int64
eccen       float64
p_black     float64
p_and       float64
mean_tr     float64
blackpix      int64
blackand      int64
wb_trans      int64
class         int64
dtype: ob

# 5) Feature and Target Definition

In this section, the predictor variables (**X**) and the target variable (**y**) are defined for all validation datasets.

For binary datasets:

- the target is converted to integer form when already binary
- label encoding is used only if necessary

For multi-class datasets:

- the target is encoded into contiguous numerical labels where needed
- all remaining columns are used as input features

After defining the feature matrices and target variables, the feature types are identified.

All validation datasets are treated as containing only numerical predictors.

In [4]:

# Feature and Target Definition

# --------------------------------------------------
# HTRU2
# --------------------------------------------------
y_htru2 = htru2[htru2_target_col].astype(int).reset_index(drop=True)
X_htru2 = htru2.drop(columns=[htru2_target_col]).reset_index(drop=True)

# --------------------------------------------------
# Page Blocks
# --------------------------------------------------
page_label_encoder = LabelEncoder()
y_page = pd.Series(
    page_label_encoder.fit_transform(page_blocks[page_target_col]),
    name="class"
)
X_page = page_blocks.drop(columns=[page_target_col]).reset_index(drop=True)

# --------------------------------------------------
# Diabetic Retinopathy Debrecen
# --------------------------------------------------
dr_target_series = diabetic_retinopathy[dr_target_col].copy()
dr_unique_values = sorted(pd.Series(dr_target_series).dropna().unique().tolist())

if set(dr_unique_values) == {0, 1}:
    y_dr = dr_target_series.astype(int).reset_index(drop=True)
else:
    dr_label_encoder = LabelEncoder()
    y_dr = pd.Series(
        dr_label_encoder.fit_transform(dr_target_series),
        name="class"
    )

X_dr = diabetic_retinopathy.drop(columns=[dr_target_col]).reset_index(drop=True)

# --------------------------------------------------
# Glass Identification
# --------------------------------------------------
glass_label_encoder = LabelEncoder()
y_glass = pd.Series(
    glass_label_encoder.fit_transform(glass[glass_target_col]),
    name="class"
)
X_glass = glass.drop(columns=[glass_target_col]).reset_index(drop=True)


# Feature type identification

categorical_features_htru2 = []
numerical_features_htru2 = X_htru2.columns.tolist()

categorical_features_page = []
numerical_features_page = X_page.columns.tolist()

categorical_features_dr = []
numerical_features_dr = X_dr.columns.tolist()

categorical_features_glass = []
numerical_features_glass = X_glass.columns.tolist()


# Verification

print("\nHTRU2 feature matrix:", X_htru2.shape)
print("HTRU2 target:", y_htru2.shape)

print("\nPage Blocks feature matrix:", X_page.shape)
print("Page Blocks target:", y_page.shape)

print("\nDiabetic Retinopathy feature matrix:", X_dr.shape)
print("Diabetic Retinopathy target:", y_dr.shape)

print("\nGlass Identification feature matrix:", X_glass.shape)
print("Glass Identification target:", y_glass.shape)

print("\nHTRU2 target distribution:")
print(y_htru2.value_counts().sort_index())

print("\nPage Blocks target distribution:")
print(y_page.value_counts().sort_index())

print("\nDiabetic Retinopathy target distribution:")
print(y_dr.value_counts().sort_index())

print("\nGlass Identification target distribution:")
print(y_glass.value_counts().sort_index())


HTRU2 feature matrix: (17898, 8)
HTRU2 target: (17898,)

Page Blocks feature matrix: (5473, 10)
Page Blocks target: (5473,)

Diabetic Retinopathy feature matrix: (1151, 19)
Diabetic Retinopathy target: (1151,)

Glass Identification feature matrix: (214, 9)
Glass Identification target: (214,)

HTRU2 target distribution:
class
0    16259
1     1639
Name: count, dtype: int64

Page Blocks target distribution:
class
0    4913
1     329
2      28
3      88
4     115
Name: count, dtype: int64

Diabetic Retinopathy target distribution:
Class
0    540
1    611
Name: count, dtype: int64

Glass Identification target distribution:
class
0    70
1    76
2    17
3    13
4     9
5    29
Name: count, dtype: int64


## 6) Dataset Characterization for Framework Application

This section characterizes each validation dataset according to the model selection framework developed in the thesis.

For each dataset, the following properties are computed:

- number of rows
- number of features
- class counts
- class proportions
- imbalance ratio
- inferred class type
- framework condition used
- primary recommendation
- alternatives

The framework rules applied are:

**Dataset size conditions**
- small dataset size: `N <= 2000`
- medium dataset size: `2001 <= N <= 6000`
- large dataset size: `N > 6000`

**Imbalance condition**
- pronounced imbalance: `IR >= 2.5`

**Binary classification framework**
- pronounced imbalance -> primary: CatBoost; alternatives: LightGBM, XGBoost
- small dataset size -> primary: XGBoost; alternative: CatBoost
- medium dataset size -> primary: CatBoost; alternative: XGBoost
- large dataset size -> primary: CatBoost; alternatives: Gradient Boosting, LightGBM

**Multi-class classification framework**
- all dataset sizes -> primary: CatBoost; alternatives: XGBoost, LightGBM
- pronounced imbalance -> primary: CatBoost; alternative: LightGBM

In [5]:
import pandas as pd

# ==================================================
# Core Logic (Framework-aligned)
# ==================================================

def compute_imbalance_ratio(y):
    counts = pd.Series(y).value_counts()
    return counts.max() / counts.min()

def infer_class_type(y):
    return "Binary" if pd.Series(y).nunique() == 2 else "Multi-class"

def framework_recommendation(class_type, n_rows, ir):
    if class_type == "Binary":
        if ir >= 2.5:
            return {
                "condition": "Pronounced imbalance (IR ≥ 2.5)",
                "primary": "CatBoost",
                "alts": "LightGBM, XGBoost"
            }
        elif n_rows <= 2000:
            return {
                "condition": "Small dataset size (N ≤ 2000)",
                "primary": "XGBoost",
                "alts": "CatBoost"
            }
        elif n_rows <= 6000:
            return {
                "condition": "Medium dataset size (2001 ≤ N ≤ 6000)",
                "primary": "CatBoost",
                "alts": "XGBoost"
            }
        else:
            return {
                "condition": "Large dataset size (N > 6000)",
                "primary": "CatBoost",
                "alts": "Gradient Boosting, LightGBM"
            }
    else:
        if ir >= 2.5:
            return {
                "condition": "Pronounced imbalance (IR ≥ 2.5)",
                "primary": "CatBoost",
                "alts": "LightGBM"
            }
        else:
            return {
                "condition": "All dataset sizes",
                "primary": "CatBoost",
                "alts": "XGBoost, LightGBM"
            }

def format_class_distribution(y):
    props = pd.Series(y).value_counts(normalize=True).sort_index()
    return " | ".join([f"C{cls}({p:.0%})" for cls, p in props.items()])

def characterize_dataset(X, y_raw, dataset_name):
    n_rows, n_features = X.shape
    class_type = infer_class_type(y_raw)
    ir = compute_imbalance_ratio(y_raw)
    fw = framework_recommendation(class_type, n_rows, ir)

    return {
        "Dataset": dataset_name,
        "Rows": int(n_rows),
        "Features": int(n_features),
        "Type": class_type,
        "Distribution": format_class_distribution(y_raw),
        "IR": round(float(ir), 2),
        "Dataset condition": fw["condition"],
        "Primary recommendation": fw["primary"],
        "Alternatives": fw["alts"]
    }

# ==================================================
# Build Validation Characterization Table
# ==================================================

data = [
    characterize_dataset(X_htru2, y_htru2, "HTRU2"),
    characterize_dataset(X_page, y_page, "Page_Blocks"),
    characterize_dataset(X_dr, y_dr, "Diabetic_Retinopathy"),
    characterize_dataset(X_glass, y_glass, "Glass_Identification")
]

validation_characterization = pd.DataFrame(data)

validation_characterization = validation_characterization[
    [
        "Dataset",
        "Rows",
        "Features",
        "Type",
        "Distribution",
        "IR",
        "Dataset condition",
        "Primary recommendation",
        "Alternatives"
    ]
]

def apply_grid_style(styler):
    styler.set_caption("Validation Dataset Characterization for Framework Application")
    styler.hide(axis="index")

    styler.format({
        "Rows": "{:,}",
        "Features": "{:,.0f}",
        "IR": "{:.2f}"
    })

    styler.set_properties(
        subset=["Primary recommendation"],
        **{"font-weight": "bold"}
    )

    styler.set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#f2f2f2"),
                ("color", "black"),
                ("border", "1px solid #333"),
                ("text-align", "center"),
                ("font-weight", "bold"),
                ("padding", "8px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #999"),
                ("text-align", "center"),
                ("padding", "8px"),
                ("font-size", "13px"),
                ("font-family", "Arial, sans-serif")
            ]
        },
        {
            "selector": "tr:nth-child(even)",
            "props": [
                ("background-color", "#fafafa")
            ]
        },
        {
            "selector": ".col4",
            "props": [
                ("min-width", "180px"),
                ("text-align", "left")
            ]
        },
        {
            "selector": ".col6",
            "props": [
                ("min-width", "220px"),
                ("text-align", "left")
            ]
        },
        {
            "selector": ".col8",
            "props": [
                ("min-width", "180px"),
                ("text-align", "left")
            ]
        }
    ])

    return styler

display(apply_grid_style(validation_characterization.style))

Dataset,Rows,Features,Type,Distribution,IR,Dataset condition,Primary recommendation,Alternatives
HTRU2,"17,898",8,Binary,C0(91%) | C1(9%),9.92,Pronounced imbalance (IR ≥ 2.5),CatBoost,"LightGBM, XGBoost"
Page_Blocks,"5,473",10,Multi-class,C0(90%) | C1(6%) | C2(1%) | C3(2%) | C4(2%),175.46,Pronounced imbalance (IR ≥ 2.5),CatBoost,LightGBM
Diabetic_Retinopathy,"1,151",19,Binary,C0(47%) | C1(53%),1.13,Small dataset size (N ≤ 2000),XGBoost,CatBoost
Glass_Identification,214,9,Multi-class,C0(33%) | C1(36%) | C2(8%) | C3(6%) | C4(4%) | C5(14%),8.44,Pronounced imbalance (IR ≥ 2.5),CatBoost,LightGBM


# 7) Preprocessing Pipeline

Before training the models, input features must be transformed into a format suitable for machine learning algorithms.

All validation datasets contain only numerical predictors, so no encoding is required.

Preprocessing is implemented using **scikit-learn’s ColumnTransformer**, which allows transformations to be integrated into a **pipeline**. This ensures that preprocessing is applied consistently during cross-validation and prevents data leakage.

In [6]:
# Preprocessing Pipeline

def build_passthrough_preprocessor(numerical_features):
    return ColumnTransformer(
        transformers=[
            ("num", "passthrough", numerical_features)
        ]
    )

preprocessor_htru2 = build_passthrough_preprocessor(numerical_features_htru2)
preprocessor_page = build_passthrough_preprocessor(numerical_features_page)
preprocessor_dr = build_passthrough_preprocessor(numerical_features_dr)
preprocessor_glass = build_passthrough_preprocessor(numerical_features_glass)

print("Preprocessors defined.")

Preprocessors defined.


# 8) Model Definitions

This section initializes the machine learning algorithms evaluated in the validation study.

At this stage, only model objects are created. No training is performed yet.

Because the validation includes both binary and multi-class datasets, separate model dictionaries are defined so that task-specific settings are handled correctly.

In [7]:
# Model Definitions

def get_binary_models():
    return {
        "DecisionTree": DecisionTreeClassifier(random_state=SEED),

        "RandomForest": RandomForestClassifier(
            random_state=SEED,
            n_jobs=1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            random_state=SEED,
            n_jobs=1
        ),

        "GradientBoosting": GradientBoostingClassifier(
            random_state=SEED
        ),

        "XGBoost": xgb.XGBClassifier(
            random_state=SEED,
            n_jobs=1,
            verbosity=0,
            eval_metric="logloss"
        ),

        "LightGBM": lgb.LGBMClassifier(
            random_state=SEED,
            n_jobs=1,
            verbosity=-1
        ),

        "CatBoost": cb.CatBoostClassifier(
            random_state=SEED,
            verbose=0,
            thread_count=1,
            loss_function="Logloss"
        )
    }

def get_multiclass_models(n_classes):
    return {
        "DecisionTree": DecisionTreeClassifier(random_state=SEED),

        "RandomForest": RandomForestClassifier(
            random_state=SEED,
            n_jobs=1
        ),

        "ExtraTrees": ExtraTreesClassifier(
            random_state=SEED,
            n_jobs=1
        ),

        "GradientBoosting": GradientBoostingClassifier(
            random_state=SEED
        ),

        "XGBoost": xgb.XGBClassifier(
            random_state=SEED,
            n_jobs=1,
            verbosity=0,
            objective="multi:softprob",
            num_class=n_classes,
            eval_metric="mlogloss"
        ),

        "LightGBM": lgb.LGBMClassifier(
            random_state=SEED,
            n_jobs=1,
            verbosity=-1,
            objective="multiclass",
            num_class=n_classes
        ),

        "CatBoost": cb.CatBoostClassifier(
            random_state=SEED,
            verbose=0,
            thread_count=1,
            loss_function="MultiClass"
        )
    }

models_binary = get_binary_models()
models_multiclass_page = get_multiclass_models(y_page.nunique())
models_multiclass_glass = get_multiclass_models(y_glass.nunique())

print("Binary models initialized:", list(models_binary.keys()))
print("Multi-class models initialized:", list(models_multiclass_page.keys()))

Binary models initialized: ['DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']
Multi-class models initialized: ['DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


# 9) Hyperparameter Optimization

Hyperparameters are optimized using **Randomized Search with stratified five-fold cross-validation**.

The tuning procedure is performed once for each original validation dataset. If previously tuned hyperparameters are already available in the repository or on GitHub, they are loaded directly in order to avoid repeating the computationally expensive tuning stage.

The optimization metric is aligned with the main validation metric:

- **HTRU2** -> F1-score
- **Diabetic Retinopathy Debrecen** -> F1-score
- **Page Blocks** -> Macro-F1
- **Glass Identification** -> Macro-F1

In [8]:

# Hyperparameter Optimization

param_distributions = {

    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "RandomForest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "ExtraTrees": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },

    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__max_depth": [3, 5, 7],
        "model__max_features": ["sqrt", "log2"],
        "model__min_samples_split": [2, 5]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
        "model__max_depth": [3, 5, 7]
    },

    "LightGBM": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__num_leaves": [31, 50, 70],
        "model__feature_fraction": [0.8, 1.0]
    },

    "CatBoost": {
        "model__iterations": [200, 400, 600],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__depth": [4, 6, 8],
        "model__l2_leaf_reg": [1, 3, 5]
    }
}

def tune_models(X, y, preprocessor, models, scoring_metric):
    best_params = {}

    for model_name, model in models.items():
        print(f"Tuning {model_name}...")

        pipeline = Pipeline([
            ("preprocessing", preprocessor),
            ("model", model)
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_distributions[model_name],
            n_iter=10,
            cv=cv,
            scoring=scoring_metric,
            n_jobs=1,
            random_state=SEED,
            verbose=0
        )

        search.fit(X, y)

        best_params[model_name] = {
            "best_params": search.best_params_,
            "best_score": search.best_score_
        }

        print("Best score:", round(search.best_score_, 4))

    return best_params

def load_from_github(url):
    try:
        r = requests.get(url)
        if r.status_code == 200:
            print(f"🌐 Loaded from GitHub: {url}")
            return r.json()
        else:
            print(f"❌ GitHub file not found: {url}")
            return None
    except Exception as e:
        print(f"⚠️ GitHub load failed: {e}")
        return None

dataset_specs = {
    "HTRU2": {
        "X": X_htru2,
        "y": y_htru2,
        "preprocessor": preprocessor_htru2,
        "models": models_binary,
        "scoring_metric": "f1",
        "config_url": "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_htru2_f1.json",
        "config_filename": "best_params_htru2_f1.json"
    },
    "Diabetic_Retinopathy_Debrecen": {
        "X": X_dr,
        "y": y_dr,
        "preprocessor": preprocessor_dr,
        "models": models_binary,
        "scoring_metric": "f1",
        "config_url": "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_diabetic_retinopathy_f1.json",
        "config_filename": "best_params_diabetic_retinopathy_f1.json"
    },
    "Page_Blocks": {
        "X": X_page,
        "y": y_page,
        "preprocessor": preprocessor_page,
        "models": models_multiclass_page,
        "scoring_metric": "f1_macro",
        "config_url": "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_page_blocks_macro_f1.json",
        "config_filename": "best_params_page_blocks_macro_f1.json"
    },
    "Glass_Identification": {
        "X": X_glass,
        "y": y_glass,
        "preprocessor": preprocessor_glass,
        "models": models_multiclass_glass,
        "scoring_metric": "f1_macro",
        "config_url": "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_glass_macro_f1.json",
        "config_filename": "best_params_glass_macro_f1.json"
    }
}

best_params_registry = {}
tuned_files_now = []

for dataset_name, spec in dataset_specs.items():
    print("\n" + "=" * 60)
    print(f"PROCESSING HYPERPARAMETERS: {dataset_name}")
    print("=" * 60)

    config_path = os.path.join(CONFIG_PATH, spec["config_filename"])

    best_params = load_from_github(spec["config_url"])

    if best_params is None and os.path.exists(config_path):
        print(f"📂 Loading {dataset_name} params from local...")
        with open(config_path) as f:
            best_params = json.load(f)

    if best_params is None:
        print(f"🚀 Running tuning for {dataset_name}...")
        best_params = tune_models(
            spec["X"],
            spec["y"],
            spec["preprocessor"],
            spec["models"],
            spec["scoring_metric"]
        )

        with open(config_path, "w") as f:
            json.dump(best_params, f, indent=4)

        tuned_files_now.append(config_path)
        print(f"💾 Saved tuned params: {config_path}")
    else:
        print(f"✅ Hyperparameters ready for {dataset_name}")

    best_params_registry[dataset_name] = best_params

# --------------------------------------------------
# Download ONLY if tuning was executed now
# --------------------------------------------------

if tuned_files_now:
    try:
        from google.colab import files
        print("\n⬇️ Downloading newly tuned hyperparameter files...")
        for path in tuned_files_now:
            files.download(path)
    except:
        print("Download skipped (not in Colab).")
else:
    print("\n📁 No download needed (loaded from GitHub/local).")


PROCESSING HYPERPARAMETERS: HTRU2
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_htru2_f1.json
✅ Hyperparameters ready for HTRU2

PROCESSING HYPERPARAMETERS: Diabetic_Retinopathy_Debrecen
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_diabetic_retinopathy_f1.json
✅ Hyperparameters ready for Diabetic_Retinopathy_Debrecen

PROCESSING HYPERPARAMETERS: Page_Blocks
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_page_blocks_macro_f1.json
✅ Hyperparameters ready for Page_Blocks

PROCESSING HYPERPARAMETERS: Glass_Identification
🌐 Loaded from GitHub: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/config/best_params_glass_macro_f1.json
✅ Hyperparameters ready for Glass_Identification

📁 No

# 10) Validation Evaluation

This section evaluates the seven machine learning algorithms on the original validation datasets.

For each dataset, all algorithms are evaluated using stratified five-fold cross-validation. The best hyperparameters obtained during the tuning stage are reused without further optimization.

The primary evaluation metrics are:

- **HTRU2** -> F1-score
- **Diabetic Retinopathy Debrecen** -> F1-score
- **Page Blocks** -> Macro-F1
- **Glass Identification** -> Macro-F1

Additional metrics are also reported:

- binary datasets: Precision, Recall, Balanced Accuracy
- multi-class datasets: Macro Precision, Macro Recall, Balanced Accuracy

The average **training time per cross-validation fold** and the **trained model size** are also recorded.

In [9]:
# VALIDATION RESULTS

VALIDATION_RESULTS_URL = "https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/validation_results_framework_4datasets.csv"

validation_results_path = os.path.join(
    RESULTS_PATH,
    "validation_results_framework_4datasets.csv"
)

print("🌐 Checking GitHub for validation results...")

validation_results_df = None
ran_now = False

# --------------------------------------------------
# Helper: normalize old/new column names and dataset names
# --------------------------------------------------
def normalize_validation_results(df):
    rename_map = {
        "framework_branch": "dataset_condition",
        "Framework branch": "dataset_condition",
        "Framework condition": "dataset_condition",
        "primary": "primary_recommendation",
        "Primary recommendation": "primary_recommendation",
        "alts": "alternatives",
        "Alternatives": "alternatives"
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    if "dataset" in df.columns:
        df["dataset"] = df["dataset"].replace({
            "Diabetic_Retinopathy_Debrecen": "Diabetic_Retinopathy"
        })

    return df

# --------------------------------------------------
# STEP 1 — GitHub
# --------------------------------------------------
try:
    r = requests.get(VALIDATION_RESULTS_URL)

    if r.status_code == 200:
        with open(validation_results_path, "wb") as f:
            f.write(r.content)

        validation_results_df = pd.read_csv(validation_results_path)
        validation_results_df = normalize_validation_results(validation_results_df)

        print("✅ Loaded from GitHub")
        print("Source:", VALIDATION_RESULTS_URL)
    else:
        print("❌ GitHub file not available")
except:
    print("⚠️ GitHub error")

# --------------------------------------------------
# STEP 2 — Local
# --------------------------------------------------
if validation_results_df is None and os.path.exists(validation_results_path):
    print("📂 Loading validation results from local...")
    validation_results_df = pd.read_csv(validation_results_path)
    validation_results_df = normalize_validation_results(validation_results_df)

# --------------------------------------------------
# STEP 3 — Run validation if missing
# --------------------------------------------------
if validation_results_df is None:

    print("🚀 Running framework validation...")

    results_validation = []

    def evaluate_dataset(
        X,
        y,
        dataset_name,
        best_params,
        preprocessor,
        models,
        scoring,
        class_type
    ):
        fw_row = validation_characterization[
            validation_characterization["Dataset"] == dataset_name
        ].iloc[0]

        for model_name, base_model in models.items():

            model = clone(base_model)

            if model_name in best_params:
                tuned_params = {
                    k.replace("model__", ""): v
                    for k, v in best_params[model_name]["best_params"].items()
                }
                model.set_params(**tuned_params)

            pipeline = Pipeline([
                ("preprocessing", preprocessor),
                ("model", model)
            ])

            cv_results = cross_validate(
                pipeline,
                X,
                y,
                cv=cv,
                scoring=scoring,
                n_jobs=1
            )

            pipeline.fit(X, y)

            row = {
                "dataset": dataset_name,
                "class_type": class_type,
                "dataset_condition": fw_row["Dataset condition"],
                "primary_recommendation": fw_row["Primary recommendation"],
                "alternatives": fw_row["Alternatives"],
                "model": model_name,
                "balanced_accuracy": cv_results["test_balanced_accuracy"].mean(),
                "training_time": cv_results["fit_time"].mean(),
                "model_size_kb": get_model_size_kb(pipeline.named_steps["model"])
            }

            if class_type == "binary":
                row["f1"] = cv_results["test_f1"].mean()
                row["precision"] = cv_results["test_precision"].mean()
                row["recall"] = cv_results["test_recall"].mean()
            else:
                row["macro_f1"] = cv_results["test_macro_f1"].mean()
                row["macro_precision"] = cv_results["test_macro_precision"].mean()
                row["macro_recall"] = cv_results["test_macro_recall"].mean()

            results_validation.append(row)

    scoring_binary = {
        "f1": "f1",
        "precision": "precision",
        "recall": "recall",
        "balanced_accuracy": "balanced_accuracy"
    }

    scoring_multiclass = {
        "macro_f1": "f1_macro",
        "macro_precision": "precision_macro",
        "macro_recall": "recall_macro",
        "balanced_accuracy": "balanced_accuracy"
    }

    evaluate_dataset(
        X_htru2,
        y_htru2,
        "HTRU2",
        best_params_registry["HTRU2"],
        preprocessor_htru2,
        models_binary,
        scoring_binary,
        "binary"
    )

    evaluate_dataset(
        X_dr,
        y_dr,
        "Diabetic_Retinopathy",
        best_params_registry["Diabetic_Retinopathy_Debrecen"],
        preprocessor_dr,
        models_binary,
        scoring_binary,
        "binary"
    )

    evaluate_dataset(
        X_page,
        y_page,
        "Page_Blocks",
        best_params_registry["Page_Blocks"],
        preprocessor_page,
        models_multiclass_page,
        scoring_multiclass,
        "multi-class"
    )

    evaluate_dataset(
        X_glass,
        y_glass,
        "Glass_Identification",
        best_params_registry["Glass_Identification"],
        preprocessor_glass,
        models_multiclass_glass,
        scoring_multiclass,
        "multi-class"
    )

    validation_results_df = pd.DataFrame(results_validation)
    validation_results_df.to_csv(validation_results_path, index=False)
    ran_now = True

    print("💾 Validation results saved:", validation_results_path)

if ran_now:
    try:
        from google.colab import files
        print("⬇️ Downloading validation results...")
        files.download(validation_results_path)
    except:
        print("Download skipped")
else:
    print("📁 No download needed (loaded from GitHub/local)")

print("\nValidation results columns:")
print(validation_results_df.columns.tolist())
print("\nValidation dataset names:")
print(sorted(validation_results_df["dataset"].unique().tolist()))

🌐 Checking GitHub for validation results...
✅ Loaded from GitHub
Source: https://raw.githubusercontent.com/Ilaha-Habibova/Tree-algorithms-dataset-characteristics/main/results/validation_results_framework_4datasets.csv
📁 No download needed (loaded from GitHub/local)

Validation results columns:
['dataset', 'class_type', 'dataset_condition', 'primary_recommendation', 'alternatives', 'model', 'balanced_accuracy', 'training_time', 'model_size_kb', 'f1', 'precision', 'recall', 'macro_f1', 'macro_precision', 'macro_recall']

Validation dataset names:
['Diabetic_Retinopathy', 'Glass_Identification', 'HTRU2', 'Page_Blocks']


# 11) Validation Results Tables

This section presents the validation results in tabular form.

Binary datasets are sorted by **F1-score**.  
Multi-class datasets are sorted by **Macro-F1**.

In [10]:
# VALIDATION RESULTS TABLES

def show_binary_results(dataset_name):
    print(f"\n{dataset_name.upper()} VALIDATION RESULTS")
    display(
        validation_results_df[
            validation_results_df["dataset"] == dataset_name
        ][[
            "model",
            "f1",
            "precision",
            "recall",
            "balanced_accuracy",
            "training_time",
            "model_size_kb"
        ]].round(4).sort_values(by="f1", ascending=False).reset_index(drop=True)
    )

def show_multiclass_results(dataset_name):
    print(f"\n{dataset_name.upper()} VALIDATION RESULTS")
    display(
        validation_results_df[
            validation_results_df["dataset"] == dataset_name
        ][[
            "model",
            "macro_f1",
            "macro_precision",
            "macro_recall",
            "balanced_accuracy",
            "training_time",
            "model_size_kb"
        ]].round(4).sort_values(by="macro_f1", ascending=False).reset_index(drop=True)
    )

show_binary_results("HTRU2")
show_binary_results("Diabetic_Retinopathy")
show_multiclass_results("Page_Blocks")
show_multiclass_results("Glass_Identification")


HTRU2 VALIDATION RESULTS


,model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
0,XGBoost,0.8914,0.9323,0.8542,0.9239,0.3551,325.8945
1,LightGBM,0.8893,0.9292,0.8529,0.9232,0.4330,669.9385
2,CatBoost,0.8881,0.9301,0.8499,0.9217,3.1935,202.0752
3,RandomForest,0.8878,0.9325,0.8475,0.9206,13.3306,8673.2891
4,GradientBoosting,0.8867,0.9330,0.8450,0.9194,7.2369,972.2812
5,ExtraTrees,0.8849,0.9328,0.8420,0.9179,1.5533,23706.9814
6,DecisionTree,0.8764,0.9304,0.8285,0.9111,0.3542,5.4150



DIABETIC_RETINOPATHY VALIDATION RESULTS


,model,f1,precision,recall,balanced_accuracy,training_time,model_size_kb
0,CatBoost,0.7204,0.7381,0.7053,0.7110,1.1201,203.0215
1,XGBoost,0.7124,0.7394,0.6889,0.7074,0.2002,325.1348
2,LightGBM,0.7105,0.7205,0.7020,0.6964,0.1962,676.1230
3,GradientBoosting,0.7078,0.7259,0.6922,0.6979,0.9001,1159.4795
4,ExtraTrees,0.7070,0.7333,0.6840,0.7012,0.2073,5087.8428
5,RandomForest,0.6947,0.7142,0.6774,0.6859,0.4040,2497.4629
6,DecisionTree,0.6421,0.6540,0.6316,0.6269,0.0216,31.5107



PAGE_BLOCKS VALIDATION RESULTS


,model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
0,CatBoost,0.8745,0.8846,0.8787,0.8787,3.4972,503.8730
1,XGBoost,0.8653,0.8892,0.8527,0.8527,0.8501,1559.8877
2,ExtraTrees,0.8635,0.9119,0.8343,0.8343,0.3232,5494.7578
3,RandomForest,0.8631,0.8826,0.8578,0.8578,1.4118,2204.1406
4,LightGBM,0.8630,0.9011,0.8417,0.8417,2.3237,7608.2236
5,GradientBoosting,0.8581,0.8783,0.8499,0.8499,10.0979,3956.7002
6,DecisionTree,0.8310,0.8570,0.8329,0.8329,0.0523,20.5400



GLASS_IDENTIFICATION VALIDATION RESULTS


,model,macro_f1,macro_precision,macro_recall,balanced_accuracy,training_time,model_size_kb
0,CatBoost,0.7386,0.8138,0.7182,0.7182,7.5460,5572.5811
1,LightGBM,0.7280,0.7464,0.7287,0.7287,0.1450,1188.8398
2,RandomForest,0.7233,0.7745,0.7295,0.7295,0.2346,696.5000
3,GradientBoosting,0.7209,0.7838,0.7029,0.7029,2.4228,2835.5146
4,XGBoost,0.6981,0.7282,0.7075,0.7075,0.1794,1143.8076
5,ExtraTrees,0.6697,0.6876,0.6775,0.6775,0.4057,3027.6797
6,DecisionTree,0.6636,0.6881,0.6737,0.6737,0.0062,6.9463


# 12) Framework Consistency Summary

This section compares the observed best-performing model on each validation dataset with the model recommended by the framework.

Interpretation:

- **Confirmed** -> the observed best model matches the framework’s primary recommendation
- **Supported** -> the observed best model matches one of the framework alternatives
- **Not supported** -> the observed best model matches neither the primary recommendation nor the listed alternatives

In [11]:
import pandas as pd

# ==================================================
# Framework Validation Summary Logic
# ==================================================

summary_rows = []

def summarize_framework_consistency(dataset_name, score_column):
    # Sort results to find the empirical winner
    results = (
        validation_results_df[validation_results_df["dataset"] == dataset_name]
        .copy()
        .sort_values(by=score_column, ascending=False)
        .reset_index(drop=True)
    )

    if results.empty:
        raise ValueError(f"No rows found in validation_results_df for dataset: {dataset_name}")

    # Reference the characterization table for framework "promises"
    fw_row = validation_characterization[
        validation_characterization["Dataset"] == dataset_name
    ].iloc[0]

    best_model = results.loc[0, "model"]
    primary = fw_row["Primary recommendation"]
    alternatives_raw = fw_row["Alternatives"]
    alternatives_list = [x.strip() for x in alternatives_raw.split(",")]
    best_score = round(float(results.loc[0, score_column]), 4)

    # Validation Logic
    if best_model == primary:
        status = "Confirmed"
    elif best_model in alternatives_list:
        status = "Supported"
    else:
        status = "Not supported"

    return {
        "Dataset": dataset_name,
        "Dataset condition": fw_row["Dataset condition"],
        "Primary recommendation": primary,
        "Alternatives": alternatives_raw,
        "Observed best model": best_model,
        "Observed best score": best_score,
        "Validation outcome": status
    }

# Generate Data
summary_rows.append(summarize_framework_consistency("HTRU2", "f1"))
summary_rows.append(summarize_framework_consistency("Page_Blocks", "macro_f1"))
summary_rows.append(summarize_framework_consistency("Diabetic_Retinopathy", "f1"))
summary_rows.append(summarize_framework_consistency("Glass_Identification", "macro_f1"))

validation_summary_df = pd.DataFrame(summary_rows)

# ==================================================
# Advanced Styling & Design
# ==================================================

def apply_summary_style(styler):
    styler.set_caption("Framework Validation Summary: Observed vs. Recommended")
    styler.hide(axis="index")

    # Format scores
    styler.format({"Observed best score": "{:.4f}"})

    # 1. Semantic Text Coloring for Validation Outcome
    def color_outcome_text(val):
        if val == "Confirmed": return "color: #28a745; font-weight: bold;"
        if val == "Supported": return "color: #fd7e14; font-weight: bold;"
        if val == "Not supported": return "color: #dc3545; font-weight: bold;"
        return ""

    # 2. Strategic Cell Highlighting (Highlighting the "Source" of the best model)
    def highlight_match_source(row):
        styles = [''] * len(row)
        best = row["Observed best model"]
        primary = row["Primary recommendation"]
        alternatives = [x.strip() for x in row["Alternatives"].split(",")]

        p_idx = row.index.get_loc("Primary recommendation")
        a_idx = row.index.get_loc("Alternatives")

        if best == primary:
            # Highlight Primary in Green if it won
            styles[p_idx] = 'background-color: #d4edda; border: 2px solid #28a745 !important; font-weight: bold;'
        elif best in alternatives:
            # Highlight Alternatives in Orange if an alt won
            styles[a_idx] = 'background-color: #fff3cd; border: 2px solid #fd7e14 !important; font-weight: bold;'
        return styles

    # Apply Logic
    styler.applymap(color_outcome_text, subset=["Validation outcome"])
    styler.apply(highlight_match_source, axis=1)

    # Bold the Observed Best Model for clarity
    styler.set_properties(subset=["Observed best model"], **{"font-weight": "bold"})

    # 3. Robust Grid CSS
    styler.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#f2f2f2'), ('color', 'black'),
            ('border', '1px solid #333'), ('text-align', 'center'),
            ('font-weight', 'bold'), ('padding', '10px')
        ]},
        {'selector': 'td', 'props': [
            ('border', '1px solid #999'), ('text-align', 'center'),
            ('padding', '8px'), ('font-size', '13px'), ('font-family', 'Arial, sans-serif')
        ]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#fafafa')]},
        {'selector': '.col1', 'props': [('min-width', '220px'), ('text-align', 'left')]},
        {'selector': '.col3', 'props': [('min-width', '180px'), ('text-align', 'left')]}
    ])

    return styler

print("Framework validation summary:")
display(apply_summary_style(validation_summary_df.style))

Framework validation summary:


Dataset,Dataset condition,Primary recommendation,Alternatives,Observed best model,Observed best score,Validation outcome
HTRU2,Pronounced imbalance (IR ≥ 2.5),CatBoost,"LightGBM, XGBoost",XGBoost,0.8914,Supported
Page_Blocks,Pronounced imbalance (IR ≥ 2.5),CatBoost,LightGBM,CatBoost,0.8745,Confirmed
Diabetic_Retinopathy,Small dataset size (N ≤ 2000),XGBoost,CatBoost,CatBoost,0.7204,Supported
Glass_Identification,Pronounced imbalance (IR ≥ 2.5),CatBoost,LightGBM,CatBoost,0.7386,Confirmed
